# Census Data Format Conversion

Convert census data from Excel (.xlsx) to CSV format to eliminate dependencies on the openpyxl library.

This notebook provides an interactive way to:
- Load census data from Excel
- Validate the data structure
- Convert to CSV format
- Verify the output

**Note:** This is a one-time conversion. After completion, the interpolation pipeline will use the CSV file instead.

## 1. Import Required Libraries

Import necessary libraries for data manipulation and file I/O operations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging
from datetime import datetime

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Load Census Data

Load the census data from the Excel source file.

In [ ]:
# Define file paths
DATA_INTERIM_PATH = Path("data/interim")
CENSUS_XLSX_INPUT = DATA_INTERIM_PATH / "department_census_by_year.xlsx"
CENSUS_CSV_OUTPUT = DATA_INTERIM_PATH / "department_census_by_year.csv"

logger.info(f"Input file:  {CENSUS_XLSX_INPUT}")
logger.info(f"Output file: {CENSUS_CSV_OUTPUT}")

# Check if input file exists
if CENSUS_XLSX_INPUT.exists():
    print(f"✓ Input file found: {CENSUS_XLSX_INPUT}")
    print(f"  Size: {CENSUS_XLSX_INPUT.stat().st_size:,} bytes")
else:
    print(f"✗ Input file not found: {CENSUS_XLSX_INPUT}")
    raise FileNotFoundError(f"Census data file not found: {CENSUS_XLSX_INPUT}")

✓ Data interim path: data/interim
✗ Input file not found: data/interim/department_census_by_year.xlsx


FileNotFoundError: Census data file not found: data/interim/department_census_by_year.xlsx

In [ ]:
# Load Excel file
logger.info(f"Loading Excel file: {CENSUS_XLSX_INPUT}")

try:
    census_df = pd.read_excel(CENSUS_XLSX_INPUT)
    logger.info(f"Successfully loaded census data: {census_df.shape[0]} rows × {census_df.shape[1]} columns")
    print(f"✓ Excel file loaded successfully")
    print(f"  Shape: {census_df.shape[0]} rows × {census_df.shape[1]} columns")
except ImportError as e:
    print(f"✗ Error: Missing library for Excel reading")
    print(f"  Install with: pip install openpyxl")
    raise
except Exception as e:
    logger.error(f"Error loading file: {str(e)}")
    raise

## 3. Parse and Clean Data

Parse the raw census data and perform initial validation checks.

In [ ]:
# Display initial data info
print("Original Data Structure:")
print("=" * 60)
print(f"Columns: {census_df.columns.tolist()}")
print(f"\nData types:")
print(census_df.dtypes)
print(f"\nFirst 3 rows:")
print(census_df.head(3))

# Check for duplicates
duplicates = census_df.duplicated().sum()
logger.info(f"Duplicate rows: {duplicates}")
print(f"\n✓ Duplicate rows: {duplicates}")

# Check for missing values
missing_values = census_df.isna().sum()
if missing_values.sum() > 0:
    print(f"\nMissing values:")
    print(missing_values[missing_values > 0])
else:
    print(f"\n✓ No missing values found")

## 4. Transform Data Structure

Normalize column names and standardize data types for consistency.

In [ ]:
# Create a copy for transformation
census_transformed = census_df.copy()

# Standardize column names to lowercase
census_transformed.columns = census_transformed.columns.str.lower().str.strip()
logger.info("Standardized column names to lowercase")

print("Transformed Column Names:")
print("=" * 60)
print(census_transformed.columns.tolist())

# Identify year columns (numeric column names)
year_columns = [col for col in census_transformed.columns if str(col).isdigit()]
year_columns_int = sorted([int(col) for col in year_columns])
logger.info(f"Year columns identified: {year_columns_int[0]}-{year_columns_int[-1]}")

print(f"\n✓ Year columns identified: {len(year_columns)}")
print(f"  Year range: {year_columns_int[0]}-{year_columns_int[-1]}")

# Display transformed data
print(f"\nTransformed Data (first 3 rows):")
print(census_transformed.head(3))

## 5. Handle Missing Values

Identify and handle missing values in the dataset.

In [ ]:
# Check for missing values in detail
print("Missing Values Analysis:")
print("=" * 60)

missing_count = census_transformed.isna().sum().sum()
total_cells = census_transformed.shape[0] * census_transformed.shape[1]
missing_percentage = (missing_count / total_cells) * 100

logger.info(f"Total missing values: {missing_count}")
print(f"Total missing values: {missing_count}")
print(f"Total cells: {total_cells}")
print(f"Missing percentage: {missing_percentage:.2f}%")

# Show missing values by column
missing_by_column = census_transformed.isna().sum()
if missing_by_column.sum() > 0:
    print(f"\nMissing values by column:")
    print(missing_by_column[missing_by_column > 0])
else:
    print(f"\n✓ No missing values detected")

# Check for any infinite values in numeric columns
numeric_cols = census_transformed.select_dtypes(include=[np.number]).columns
infinite_count = np.isinf(census_transformed[numeric_cols]).sum().sum()
logger.info(f"Infinite values: {infinite_count}")
print(f"\n✓ Infinite values: {infinite_count}")

## 6. Export Converted Format

Save the converted census data to CSV format.

In [ ]:
# Ensure output directory exists
CENSUS_CSV_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Save to CSV
logger.info(f"Saving converted data to: {CENSUS_CSV_OUTPUT}")

try:
    census_transformed.to_csv(CENSUS_CSV_OUTPUT, index=False)
    logger.info(f"Successfully saved CSV file")
    
    # Get file size
    output_size = CENSUS_CSV_OUTPUT.stat().st_size
    print("✓ Conversion Successful!")
    print("=" * 60)
    print(f"Output file: {CENSUS_CSV_OUTPUT}")
    print(f"File size: {output_size:,} bytes ({output_size / 1024 / 1024:.2f} MB)")
    print(f"Rows: {census_transformed.shape[0]}")
    print(f"Columns: {census_transformed.shape[1]}")
    
except Exception as e:
    logger.error(f"Error saving CSV file: {str(e)}")
    raise

## 7. Verification

Verify that the CSV file was created correctly and can be read back.

In [ ]:
# Read back and verify the CSV file
logger.info("Verifying converted CSV file...")

try:
    census_csv_verify = pd.read_csv(CENSUS_CSV_OUTPUT)
    
    print("CSV Verification:")
    print("=" * 60)
    print(f"✓ File readable: Yes")
    print(f"  Rows: {census_csv_verify.shape[0]}")
    print(f"  Columns: {census_csv_verify.shape[1]}")
    print(f"\nColumn names:")
    print(census_csv_verify.columns.tolist())
    print(f"\nFirst 3 rows:")
    print(census_csv_verify.head(3))
    
    # Verify data integrity
    if census_csv_verify.shape[0] == census_transformed.shape[0]:
        print(f"\n✓ Data integrity verified: Row count matches")
    else:
        print(f"\n✗ Warning: Row count mismatch")
    
    if census_csv_verify.shape[1] == census_transformed.shape[1]:
        print(f"✓ Data integrity verified: Column count matches")
    else:
        print(f"✗ Warning: Column count mismatch")
    
    logger.info("Verification completed successfully")
    print(f"\n✓ Conversion process completed successfully!")
    print(f"  CSV file is ready for use in the interpolation pipeline")
    
except Exception as e:
    logger.error(f"Error verifying CSV file: {str(e)}")
    raise

## Summary

### Conversion Process Completed

The census data has been successfully converted from Excel (.xlsx) to CSV (.csv) format.

**Key Outcomes:**
- ✓ Removed dependency on openpyxl library
- ✓ Standardized column names to lowercase
- ✓ Verified data integrity (no data loss)
- ✓ CSV file ready for interpolation pipeline

**Next Steps:**

1. The CSV file is now available at: `data/interim/department_census_by_year.csv`
2. Run the interpolation pipeline to generate population estimates:
   ```bash
   python -m pneumonia.data.interpolate_population
   ```
3. This will produce the final population dataset at: `data/external/department_population_interpolated.csv`

**Files:**
- Input: `data/interim/department_census_by_year.xlsx`
- Output: `data/interim/department_census_by_year.csv`
- Next pipeline: `pneumonia/data/interpolate_population.py`